# XGBoost — Walmart Store Sales Forecasting

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## MLflow / DagsHub ინიციალიზაცია და GitHub-დან preprocessing-ის იმპორტი

In [3]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

!wget -q -O preprocessing.py "https://raw.githubusercontent.com/IrakliZerekidze/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/main/preprocessing.py"
from preprocessing import run_pipeline, weighted_mae, THANKSGIVING, CHRISTMAS

In [17]:
%pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='izere23',
             repo_name='ML-Final-Walmart-Recruiting-Store-Sales-Forecasting',
             mlflow=True)

import mlflow
import mlflow.sklearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 93.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 93.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=146c65ed-21e8-48dc-8b90-38e2ace06bc9&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=951093078d79ee65f6ab4f5a788fe2b6c2a03729baff80dc66103396a1c2d9a1




Accessing as izere23

Initialized MLflow to track repo "izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting"

Repository izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting initialized!

## მონაცემების ჩატვირთვა და train/validation split

ვიყენებტ `run_pipeline`-ს: `merge` → `grid-fill` → `markdown flags` → `calendar features`. train/validation ხდება time split-ით

In [4]:
train_part, valid_part, train_full, test_full = run_pipeline(
    data_dir="/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting",
    out_dir="/kaggle/working/processed",
)

for df in (train_part, valid_part, test_full):
    if "IsHoliday" not in df.columns and "IsHoliday_x" in df.columns:
        df.rename(columns={"IsHoliday_x": "IsHoliday"}, inplace=True)
        df.drop(columns=["IsHoliday_y"], inplace=True)

TARGET = "Weekly_Sales"
train = train_part.dropna(subset=[TARGET]).copy()
valid = valid_part.dropna(subset=[TARGET]).copy()

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_val,   y_val   = valid.drop(columns=[TARGET]), valid[TARGET]

print(X_train.shape, X_val.shape)

Expected rows if no gaps: 428409
Actual rows: 380107
Missing (Store,Dept,Date) combos filled in: 48302
Expected rows if no gaps: 476333
Actual rows: 421570
Missing (Store,Dept,Date) combos filled in: 54763
Saved parquet files to /kaggle/working/processed
(380107, 32) (41463, 31)


## Preprocessing და Feature Engineering transformer-ები

In [5]:
class WalmartPreprocessor(BaseEstimator, TransformerMixin):
    """Drop target-leak columns + Date, cast identity columns to category."""
    def __init__(self):
        self.drop_cols = ["Weekly_Sales_clipped", "is_negative_sales", "Date", "was_grid_filled"]
        self.cat_cols = ["Store", "Dept", "Type"]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[c for c in self.drop_cols if c in X.columns])
        for c in self.cat_cols:
            if c in X.columns:
                X[c] = X[c].astype("category")
        if "IsHoliday" in X.columns:
            X["IsHoliday"] = X["IsHoliday"].astype(int)
        return X

In [ ]:
class StoreDeptTargetEncoder(BaseEstimator, TransformerMixin):
    """Add mean historical sales per (Store, Dept), learned on TRAIN only.
       Smoothed toward the global mean so small groups don't get noisy values."""
    def __init__(self, m=10):
        self.m = m

    def fit(self, X, y):
        df = X[["Store", "Dept"]].copy()
        df["y"] = np.asarray(y)
        self.global_mean_ = df["y"].mean()
        stats = df.groupby(["Store", "Dept"])["y"].agg(["mean", "count"])
        smoothed = (stats["count"] * stats["mean"] + self.m * self.global_mean_) \
                   / (stats["count"] + self.m)
        self.mapping_ = smoothed.to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        keys = list(zip(X["Store"], X["Dept"]))
        X["store_dept_te"] = [self.mapping_.get(k, self.global_mean_) for k in keys]
        return X

### Lag52Adder — გასული წლის იმავე კვირის გაყიდვა

ამატებს `lag_52`-ს — იმავე `(Store, Dept)`-ის გაყიდვას **52 კვირის წინ** (გასული წლის იმავე კვირა), მოძებნილს კალენდარული თარიღით. იჭერს წლიურ სეზონურობას (გასული Christmas → ამ Christmas-ს).

**Leakage-safe**: history მხოლოდ train-ისგან შენდება. კრიტიკულია, რომ ეს lag test-ზეც არსებობს — გასული წელი ყოველთვის history-შია, განსხვავებით მოკლე lag-ებისგან (`lag_1` და ა.შ.), რომლებიც future block-ზე ვერ ითვლება.

In [7]:
class Lag52Adder(BaseEstimator, TransformerMixin):
    """Same-week-last-year sales (52 weeks back), looked up by calendar date.
       History built from TRAIN only (fit), so val/test never see their own targets."""
    def fit(self, X, y):
        hist = X[["Store", "Dept", "Date"]].copy()
        hist["sales"] = np.asarray(y)
        hist = hist.drop_duplicates(subset=["Store", "Dept", "Date"])
        self.history_ = hist.set_index(["Store", "Dept", "Date"])["sales"]
        return self

    def transform(self, X):
        X = X.copy()
        lag_date = X["Date"] - pd.Timedelta(weeks=52)
        idx = pd.MultiIndex.from_arrays([X["Store"], X["Dept"], lag_date])
        X["lag_52"] = self.history_.reindex(idx).values
        return X

In [ ]:
class RollingMeanAdder(BaseEstimator, TransformerMixin):
    """Trailing 4/8-week mean sales per (Store,Dept). Uses the most recent
       available history at-or-before each row's date (works on val/test)."""
    def __init__(self, windows=(4, 8)):
        self.windows = windows

    def fit(self, X, y):
        h = X[["Store", "Dept", "Date"]].copy()
        h["sales"] = np.asarray(y)
        h = (h.drop_duplicates(["Store", "Dept", "Date"])
               .sort_values(["Store", "Dept", "Date"]))
        for w in self.windows:
            h[f"roll_{w}"] = (
                h.groupby(["Store", "Dept"])["sales"]
                 .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
            )
        self.history_ = h[["Store", "Dept", "Date"] + [f"roll_{w}" for w in self.windows]]
        return self

    def transform(self, X):
        X = X.copy()
        X["_orig_order"] = np.arange(len(X))
        cols = [f"roll_{w}" for w in self.windows]

        left = X.sort_values("Date")
        right = self.history_.sort_values("Date")

        merged = pd.merge_asof(
            left, right,
            on="Date", by=["Store", "Dept"],
            direction="backward",
        )
        merged = merged.sort_values("_orig_order").drop(columns="_orig_order")
        for c in cols:
            X[c] = merged[c].values
        X = X.drop(columns="_orig_order")
        return X

In [ ]:
from sklearn.base import RegressorMixin, clone

class LogTargetRegressor(BaseEstimator, RegressorMixin):
    """Fit on log1p(y), predict back in raw units via expm1."""
    def __init__(self, model):
        self.model = model

    def fit(self, X, y, **fit_params):
        self.model_ = clone(self.model)
        y_clipped = np.clip(np.asarray(y), 0, None)
        self.model_.fit(X, np.log1p(y_clipped), **fit_params)
        return self

    def predict(self, X):
        return np.expm1(self.model_.predict(X))

In [ ]:
class HolidayCountdownAdder(BaseEstimator, TransformerMixin):
    """Weeks-until-Thanksgiving and weeks-until-Christmas (capped at +/-8)."""
    def __init__(self):
        self.thanksgiving = THANKSGIVING
        self.christmas = CHRISTMAS

    def fit(self, X, y=None):
        return self

    def _weeks_to_nearest(self, dates, holidays):
        d = dates.values.astype("datetime64[D]")
        h = np.array(holidays.values, dtype="datetime64[D]")
        diffs = (h[None, :] - d[:, None]) / np.timedelta64(7, "D")
        nearest = diffs[np.arange(len(d)), np.abs(diffs).argmin(axis=1)]
        return np.clip(nearest, -8, 8)

    def transform(self, X):
        X = X.copy()
        X["weeks_to_thanksgiving"] = self._weeks_to_nearest(X["Date"], self.thanksgiving)
        X["weeks_to_christmas"] = self._weeks_to_nearest(X["Date"], self.christmas)
        return X

In [11]:
class LagAdder(BaseEstimator, TransformerMixin):
    """Add sales lagged by N weeks per (Store,Dept), looked up by calendar date.
       History from TRAIN only."""
    def __init__(self, lags=(52,)):
        self.lags = lags

    def fit(self, X, y):
        h = X[["Store", "Dept", "Date"]].copy()
        h["sales"] = np.asarray(y)
        self.history_ = (h.drop_duplicates(["Store", "Dept", "Date"])
                          .set_index(["Store", "Dept", "Date"])["sales"])
        return self

    def transform(self, X):
        X = X.copy()
        for L in self.lags:
            lag_date = X["Date"] - pd.Timedelta(weeks=L)
            idx = pd.MultiIndex.from_arrays([X["Store"], X["Dept"], lag_date])
            X[f"lag_{L}"] = self.history_.reindex(idx).values
        return X

In [12]:
class StoreContextAdder(BaseEstimator, TransformerMixin):
    """Each dept's average share of its store's weekly sales.
       Static per (Store,Dept), learned on TRAIN only — transfers to val/test."""
    def fit(self, X, y):
        h = X[["Store", "Dept", "Date"]].copy()
        h["sales"] = np.asarray(y)
        store_week = h.groupby(["Store", "Date"])["sales"].sum().rename("stot")
        h = h.merge(store_week, on=["Store", "Date"], how="left")
        h["share"] = h["sales"] / h["stot"].replace(0, np.nan)
        self.dept_share_ = h.groupby(["Store", "Dept"])["share"].mean()
        self.global_share_ = h["share"].mean()
        return self

    def transform(self, X):
        X = X.copy()
        keys = list(zip(X["Store"], X["Dept"]))
        X["dept_share"] = [self.dept_share_.get(k, self.global_share_) for k in keys]
        return X

In [13]:
class ExtraHolidayAdder(BaseEstimator, TransformerMixin):
    """Flags for US retail holidays beyond the competition's big four.
       Uses the nearest Friday (data is W-FRI) within a week of each date."""
    # dates per year present in the data (2010-2012 train, 2013 test)
    EASTER      = pd.to_datetime(["2010-04-04","2011-04-24","2012-04-08","2013-03-31"])
    VALENTINES  = pd.to_datetime(["2010-02-14","2011-02-14","2012-02-14","2013-02-14"])
    MOTHERS     = pd.to_datetime(["2010-05-09","2011-05-08","2012-05-13","2013-05-12"])
    JULY4       = pd.to_datetime(["2010-07-04","2011-07-04","2012-07-04","2013-07-04"])
    HALLOWEEN   = pd.to_datetime(["2010-10-31","2011-10-31","2012-10-31","2013-10-31"])
    BACK2SCHOOL = pd.to_datetime(["2010-08-20","2011-08-19","2012-08-17","2013-08-16"])

    def fit(self, X, y=None):
        return self

    def _near(self, dates, holidays, days=7):
        d = dates.values.astype("datetime64[D]")
        h = np.array(holidays.values, dtype="datetime64[D]")
        diff = np.abs(h[None, :] - d[:, None]) / np.timedelta64(1, "D")
        return (diff.min(axis=1) <= days).astype(int)

    def transform(self, X):
        X = X.copy()
        X["is_easter"]      = self._near(X["Date"], self.EASTER)
        X["is_valentines"]  = self._near(X["Date"], self.VALENTINES)
        X["is_mothers"]     = self._near(X["Date"], self.MOTHERS)
        X["is_july4"]       = self._near(X["Date"], self.JULY4)
        X["is_halloween"]   = self._near(X["Date"], self.HALLOWEEN)
        X["is_back2school"] = self._near(X["Date"], self.BACK2SCHOOL)
        return X

In [ ]:
class RichEncoder(BaseEstimator, TransformerMixin):
    """Mean, std, median sales at multiple grouping levels. Train-only, smoothed."""
    def __init__(self, m=10):
        self.m = m
        self.levels = [["Store", "Dept"], ["Dept"], ["Store"], ["Dept", "Type"]]
    def fit(self, X, y):
        df = X.copy(); df["y"] = np.asarray(y)
        self.gmean_ = df["y"].mean(); self.gstd_ = df["y"].std(); self.gmed_ = df["y"].median()
        self.maps_ = {}
        for lvl in self.levels:
            if not all(c in df.columns for c in lvl): continue
            stats = df.groupby(lvl)["y"].agg(["mean", "count", "std", "median"])
            sm = (stats["count"] * stats["mean"] + self.m * self.gmean_) / (stats["count"] + self.m)
            self.maps_["_".join(lvl)] = (lvl, sm.to_dict(),
                                         stats["std"].fillna(self.gstd_).to_dict(),
                                         stats["median"].to_dict())
        return self
    def transform(self, X):
        X = X.copy()
        for name, (lvl, mean_map, std_map, med_map) in self.maps_.items():
            keys = list(X[lvl[0]] if len(lvl) == 1 else zip(*[X[c] for c in lvl]))
            X[f"te_mean_{name}"] = [mean_map.get(k, self.gmean_) for k in keys]
            X[f"te_std_{name}"]  = [std_map.get(k, self.gstd_) for k in keys]
            X[f"te_med_{name}"]  = [med_map.get(k, self.gmed_) for k in keys]
        return X

In [ ]:
class CalendarExtras(BaseEstimator, TransformerMixin):
    ALL_HOL = pd.to_datetime([
        "2010-02-12","2011-02-11","2012-02-10","2013-02-08",
        "2010-09-10","2011-09-09","2012-09-07","2013-09-06",
        "2010-11-26","2011-11-25","2012-11-23","2013-11-29",
        "2010-12-31","2011-12-30","2012-12-28","2013-12-27"])
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy(); d = X["Date"]
        X["week_of_month"] = ((d.dt.day - 1) // 7 + 1).astype(int)
        X["is_month_end"]  = (d.dt.day >= 24).astype(int)
        X["is_quarter_end"] = ((d.dt.month.isin([3, 6, 9, 12])) & (d.dt.day >= 24)).astype(int)
        dd = d.values.astype("datetime64[D]"); h = self.ALL_HOL.values.astype("datetime64[D]")
        X["days_to_holiday"] = (np.abs(h[None, :] - dd[:, None]) / np.timedelta64(1, "D")).min(axis=1)
        return X

In [ ]:
class YoYRatioAdder(BaseEstimator, TransformerMixin):
    """lag_52 / lag_104 — year-over-year growth."""
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        if "lag_52" in X.columns and "lag_104" in X.columns:
            X["yoy_ratio"] = X["lag_52"] / X["lag_104"].replace(0, np.nan)
        return X


## საბოლოო მოდელი

In [18]:
from xgboost import XGBRegressor
from mlflow.models.signature import infer_signature

labeled = train_full.dropna(subset=[TARGET])
X_all, y_all = labeled.drop(columns=[TARGET]), labeled[TARGET]

model = XGBRegressor(
    n_estimators=2000, learning_rate=0.03, max_depth=8,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=20, reg_lambda=10.0, gamma=0.1,
    objective="reg:absoluteerror", eval_metric="mae",
    tree_method="hist", enable_categorical=True,
    random_state=42, n_jobs=-1,
)
best_pipe = Pipeline([
    ("rich_encoding", RichEncoder(m=5)),
    ("lags",          LagAdder(lags=(52, 104))),
    ("yoy",           YoYRatioAdder()),
    ("calendar",      CalendarExtras()),
    ("preprocessing", WalmartPreprocessor()),
    ("model", model),
])

# MLFlow Log

In [ ]:
with mlflow.start_run(run_name="XGBoost_L1_reg_leaves127analog_registered"):
    best_pipe.fit(X_train, y_train)
    val_pred = best_pipe.predict(X_val)
    val_wmae = weighted_mae(y_val.values, val_pred, X_val["IsHoliday"].values)
    train_wmae = weighted_mae(y_train.values, best_pipe.predict(X_train), X_train["IsHoliday"].values)
    val_mae  = float(np.mean(np.abs(y_val.values - val_pred)))
    val_rmse = float(np.sqrt(np.mean((y_val.values - val_pred) ** 2)))

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("features", "richenc_m5,lags52_104,yoy,calendar | obj=L1 | leaves127analog")
    mlflow.log_params(model.get_params())
    mlflow.log_metric("train_wmae", train_wmae)
    mlflow.log_metric("val_wmae", val_wmae)
    mlflow.log_metric("overfit_gap", val_wmae - train_wmae)
    mlflow.log_metric("val_mae", val_mae)
    mlflow.log_metric("val_rmse", val_rmse)

    best_pipe.fit(X_all, y_all)
    mlflow.log_param("registered_fit", "all_labeled_data")
    mlflow.sklearn.log_model(
        best_pipe, artifact_path="model",
        serialization_format="cloudpickle",
        registered_model_name="XGBoost_Walmart_final",
    )
print("registered")

2026/07/12 16:34:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 16:35:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'XGBoost_Walmart_final' already exists. Creating a new version of this model...
2026/07/12 16:35:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_Walmart_final, version 2
Created version '2' of model 'XGBoost_Walmart_final'.


🏃 View run XGBoost_L1_reg_leaves127analog_registered at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/0/runs/5f43a727b5ee40e6aee05bef7e2a3a0a
🧪 View experiment at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/0
registered
